# Transformer Palindrome Creator

This notebook trains a tiny transformer to generate palindromes.

Training examples are complete palindromes:

```text
[BOS] A B C C B A [EOS]
```

After training, we prompt it with a prefix such as `[BOS] A B C` and let it generate the mirrored completion.


## Prerequisites


In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Vocabulary and Data

The symbols are deliberately tiny: `A` through `H`, plus beginning/end tokens. Every generated training sequence is a palindrome, so the model learns a compact synthetic language.


In [ ]:
BOS = 0
EOS = 1
symbol_offset = 2

num_symbols = 8
half_len = 4
palindrome_len = 2 * half_len
full_len = 1 + palindrome_len + 1  # [BOS], palindrome, [EOS]
block_size = full_len - 1           # model input length during training
vocab_size = symbol_offset + num_symbols

def symbol_token(i):
    return symbol_offset + i

def token_name(t):
    t = int(t)
    if t == BOS:
        return "[BOS]"
    if t == EOS:
        return "[EOS]"
    return chr(ord("A") + t - symbol_offset)

def make_palindrome():
    left = [random.randrange(num_symbols) for _ in range(half_len)]
    symbols = left + list(reversed(left))
    return [BOS] + [symbol_token(s) for s in symbols] + [EOS]

def make_batch(batch_size):
    x = torch.empty(batch_size, block_size, dtype=torch.long)
    y = torch.empty(batch_size, block_size, dtype=torch.long)
    for b in range(batch_size):
        seq = make_palindrome()
        x[b] = torch.tensor(seq[:-1], dtype=torch.long)
        y[b] = torch.tensor(seq[1:], dtype=torch.long)
    return x.to(device), y.to(device)

sample = make_palindrome()
" ".join(token_name(t) for t in sample), vocab_size, block_size


## Tiny Transformer

The model predicts a next-token distribution at every position. Inspect the masking pattern and generation loop to determine which transformer type this is.


In [ ]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, block_size, dropout):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("attention_mask", torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))
        self.last_attention = None

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        qkv = self.qkv(x).view(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.attention_mask[:, :, :seq_len, :seq_len] == 0, float("-inf"))
        attention = scores.softmax(dim=-1)
        self.last_attention = attention.detach()

        context = attention @ v
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        return self.out(self.dropout(context))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, block_size, dropout):
        super().__init__()
        self.attn = MaskedSelfAttention(d_model, num_heads, block_size, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class TinyPalindromeTransformer(nn.Module):
    def __init__(self, vocab_size, block_size, d_model=96, num_heads=4, num_layers=3, d_ff=192, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, block_size, dropout)
            for _ in range(num_layers)
        ])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        h = self.token_embedding(x) + self.position_embedding(positions)
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln(h))


model = TinyPalindromeTransformer(vocab_size, block_size).to(device)
sum(p.numel() for p in model.parameters())


## Train

This trains on freshly generated palindromes. On a GPU it should finish very quickly; on CPU it is still small enough for a local educational run.


In [ ]:
batch_size = 512
steps = 1000
learning_rate = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
start = time.time()

for step in range(1, steps + 1):
    model.train()
    x, y = make_batch(batch_size)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0 or step == 1:
        model.eval()
        with torch.no_grad():
            vx, vy = make_batch(2048)
            v_logits = model(vx)
            token_acc = (v_logits.argmax(dim=-1) == vy).float().mean().item()
            final_half_acc = (v_logits[:, half_len:palindrome_len].argmax(dim=-1) == vy[:, half_len:palindrome_len]).float().mean().item()
        print(f"step {step:4d} | loss {loss.item():.4f} | token acc {token_acc:.3f} | mirror acc {final_half_acc:.3f}")

print(f"training time: {time.time() - start:.1f}s on {device}")


## Generate Palindromes

The generator repeatedly feeds the current sequence into the model and appends the most likely next token.


In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens, temperature=0.0):
    model.eval()
    tokens = torch.tensor([prompt], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        context = tokens[:, -block_size:]
        logits = model(context)[:, -1, :]
        if temperature == 0.0:
            next_token = logits.argmax(dim=-1, keepdim=True)
        else:
            probs = F.softmax(logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)
        if next_token.item() == EOS:
            break

    return tokens[0].tolist()

def show(seq):
    return " ".join(token_name(t) for t in seq)

for _ in range(8):
    left = [random.randrange(num_symbols) for _ in range(half_len)]
    prompt = [BOS] + [symbol_token(s) for s in left]
    generated = generate(prompt, max_new_tokens=half_len + 1)
    expected = prompt + [symbol_token(s) for s in reversed(left)] + [EOS]
    print("prompt:   ", show(prompt))
    print("generated:", show(generated))
    print("expected: ", show(expected))
    print()


## Visualize Attention

The heatmap shows one attention head in the final transformer block. Use the visible pattern to reason about what information each position can access.


In [ ]:
head_index = 2

model.eval()
x, y = make_batch(1)
with torch.no_grad():
    logits = model(x)

attention = model.blocks[-1].attn.last_attention[0, head_index].cpu()
labels = [token_name(t) for t in x[0].cpu()]

print("input:", " ".join(labels))
print("target next tokens:", " ".join(token_name(t) for t in y[0].cpu()))

print(attention)

plt.figure(figsize=(7, 6))
plt.imshow(attention, cmap="viridis", vmin=0, vmax=attention.max())
plt.xticks(range(block_size), labels, rotation=45, ha="right")
plt.yticks(range(block_size), labels)
plt.xlabel("attended token")
plt.ylabel("current token")
plt.colorbar(label="attention weight")
plt.tight_layout()
plt.show()
